# Importing Libraries

In [3]:
import os
import re
import time
import shutil
import datetime
import undetected_chromedriver as uc
import pandas as pd
import random
import concurrent.futures
import requests

from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# **Part 1:** Web Scrapping

## Initializae Variables

In [2]:
def init_driver():
    """Install ChromeDriver, initialize and return the driver instance."""
    options = webdriver.ChromeOptions()
    
    # Additional options for privacy and stability
    options.add_argument("--disable-cookies")
    options.add_argument("--incognito")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-plugins-discovery")
    options.add_argument("--start-maximized")
    
    # Use undetected_chromedriver (Remove JavaScript Signature)
    driver = uc.Chrome(options=options)
    
    # Further hide automation
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

driver = init_driver()

In [3]:
# Define a list of user agents for rotation (Technique 3: Rotating User Agent)
useragentarray = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:107.0) Gecko/20100101 Firefox/107.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:107.0) Gecko/20100101 Firefox/107.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36 Edg/108.0.1462.54",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.1 Safari/605.1.15",
]

## Getting All Categories

In [ ]:
# Before Starting note that the SimilarWeb website is sometime offline and may take a long time to load. it open in the day (Around 2:00 PM GMT) and close at night (Around 2:00 AM GMT)
def get_categories(driver, url="https://www.similarweb.com/category/"):
    """
    Load the SimilarWeb category page, extract all category links, and filter them.
    
    Returns:
      A list of tuples: (main_category, subcategory, link)
    """
    driver.delete_all_cookies()
    # Rotate User Agent
    ua = random.choice(useragentarray)
    driver.execute_cdp_cmd("Network.setUserAgentOverride", {"userAgent": ua})
    print(f"Set user agent to: {ua}")
    
    driver.get(url)
    # Avoid Patterns with random delay
    time.sleep(random.uniform(1, 3))  # Simulate human reading time
    
    try:
        # Wait for the categories list to load
        ul_element = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "tl-list"))
        )
        # Get all anchor tags from the list
        anchor_tags = ul_element.find_elements(By.TAG_NAME, "a")
        # Extract raw (link, text) pairs
        raw_categories = [(a.get_attribute("href"), a.text.strip()) for a in anchor_tags]
        
        # Group categories by main category (text before '>' if present)
        category_dict = {}
        for link, text in raw_categories:
            if ">" in text:
                main_cat = text.split(">")[0].strip()
            else:
                main_cat = text
            category_dict.setdefault(main_cat, []).append((link, text))
        
        # Filter: if multiple entries exist for a main category, only use those with '>'
        filtered_categories = []
        for main_cat, items in category_dict.items():
            if len(items) > 1:
                # Keep only subcategories (text containing '>')
                subcategories = [
                    (link, text.split(">")[-1].strip()) 
                    for link, text in items if ">" in text
                ]
                for sub_link, sub_text in subcategories:
                    filtered_categories.append((main_cat, sub_text, sub_link))
            else:
                # Single entry; use it as both main and subcategory
                filtered_categories.append((main_cat, main_cat, items[0][0]))
        
        print(f"Filtered categories count: {len(filtered_categories)}")
        return filtered_categories

    except TimeoutException:
        print("Failed to load categories list from SimilarWeb.")
        return []

# Get categories
categories = get_categories(driver)
if not categories:
    driver.quit()
    raise Exception("No Categories Retrieved")
else:
    print(f"Found {len(categories)} Categories")

Set user agent to: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36
Filtered categories count: 187
Found 187 Categories


## Setting Folder Structure

In [5]:
def manage_data_folder(base_dir):
    """
    Create a data folder for today's scraping date.
    Remove old date folders if the date has changed.
    """
    # Format today's date (using '-' because '/' is not allowed in folder names)
    today_str = datetime.datetime.now().strftime("%m-%d-%Y")
    data_dir = os.path.join(base_dir, today_str)
    
    # If today's folder doesn't exist, remove any other folders and create it
    if not os.path.exists(data_dir):
        if os.path.exists(base_dir):
            for item in os.listdir(base_dir):
                item_path = os.path.join(base_dir, item)
                if os.path.isdir(item_path) and item != today_str:
                    shutil.rmtree(item_path)
        os.makedirs(data_dir, exist_ok=True)
    return data_dir

# Manage raw data folder for today (remove old data folders if any)
base_data_folder = os.path.join(os.getcwd(), "raw data")
today_folder = manage_data_folder(base_data_folder)

## Extracting Table Data Function

In [6]:
def scrape_similarweb_data_page(driver, url, save_folder, file_name):
    """
    Navigate to the given URL (top websites page for a category), extract table data,
    validate it and save it as a CSV file in the given folder.
    
    Returns:
      True if the data was scraped and saved successfully, else False.
    """
    driver.get(url)
    try:
        # Wait for table header and body to load
        table_body = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "top-table__body"))
        )
        header = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "top-table__header"))
        )
    except TimeoutException:
        print(f"Timeout loading table data from {url}")
        return False

    # Define expected column names
    expected_columns = ['Rank', 'Website', 'Category', 'Rank Change',
                        'Avg. Visit Duration', 'Pages / Visit', 'Bounce Rate']

    # Process each row of the table
    rows = table_body.find_elements(By.CLASS_NAME, "top-table__row")
    data = []
    for row in rows:
        try:
            rank = row.find_element(By.CLASS_NAME, "tw-table__rank").text.strip()
            website = row.find_element(By.TAG_NAME, "a").find_element(By.CLASS_NAME, "tw-table__domain").text.strip()
            category = row.find_element(By.CLASS_NAME, "tw-table__category").text.strip()
            rank_change = row.find_element(By.CLASS_NAME, "app-parameter-change").text.strip()
            avg_visit_duration = row.find_element(By.CLASS_NAME, "tw-table__avg-visit-duration").text.strip()
            pages_per_visit = row.find_element(By.CLASS_NAME, "tw-table__pages-per-visit").text.strip()
            bounce_rate = row.find_element(By.CLASS_NAME, "tw-table__bounce-rate").text.strip()
            row_data = [rank, website, category, rank_change, avg_visit_duration, pages_per_visit, bounce_rate]
            data.append(row_data)
        except Exception as e:
            print(f"Error processing a row: {e}")
            continue

    if not data:
        print("No table data found or parsed.")
        return False
    
    # Create DataFrame and validate columns
    df = pd.DataFrame(data, columns=expected_columns)
    if set(expected_columns) - set(df.columns):
        print("Data validation error: missing columns in scraped data.")
        return False

    # Save DataFrame to CSV
    file_path = os.path.join(save_folder, file_name)
    df.to_csv(file_path, index=False)
    print(f"Data saved to {file_path}")
    return True

## Running The Script

In [7]:
# Navigate to the top websites page
top_websites_url = "https://www.similarweb.com/top-websites/"
driver.get(top_websites_url)

# Wait for the page to load by checking for a stable element (e.g., body)
try:
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    print("Page body loaded.")
except TimeoutException:
    raise Exception("Page took too long to load.")


# Handle potential cookie banner (adjust selector based on actual site)
try:
    cookie_button = driver.find_element(By.CSS_SELECTOR, "button.accept-cookies, #accept-cookies")
    cookie_button.click()
    print("Cookie banner dismissed.")
except NoSuchElementException:
    print("No cookie banner found.")


scraped_count = 0  # Counter for successfully scraped categories


# Loop through each filtered category option
for main_cat, subcat, cat_link in categories:
    retries = 5
    attempt = 0
    while attempt <= retries:
        attempt += 1
        try:
            # Clear cookies for the next iteration
            driver.delete_all_cookies()

            # Re-open dropdown for each iteration
            category_dropdown = WebDriverWait(driver, 20).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, ".swui-select[data-name='category'] .swui-select__button"))
            )
            category_dropdown.click()
            time.sleep(1)

            # Wait for the dropdown options to be visible
            WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, ".swui-select__options"))
            )

            # Build an XPath to locate the option with the subcategory text
            option_xpath = f"//button[contains(@class, 'swui-select__option') and .//span[text()='{subcat}']]"
            option = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, option_xpath))
            )
            option.click()
            print(f"Selected category option: {main_cat} > {subcat}")
            time.sleep(1)

            # Click the "Go" button to apply the selection
            go_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "pc-selects-group__apply-button"))
            )
            go_button.click()
            print("Clicked 'Go' button.")

            # Wait for the table data to load or detect 504 error
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "top-table__body"))
                )
                print("Table data loaded.")
            except TimeoutException:
                # Check for 504 Gateway Timeout in page source
                if "504" in driver.page_source or "Gateway Timeout" in driver.page_source:
                    print("Detected 504 Gateway Timeout. Refreshing the page...")
                    driver.refresh()
                    # Wait again for the table data after refresh
                    WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CLASS_NAME, "top-table__body"))
                    )
                    print("Table data loaded after refresh.")
                else:
                    raise  # Re-raise the exception if it's not a 504 error

            # Parse the current URL to derive folder names
            current_url = driver.current_url
            parsed = urlparse(current_url)
            path_parts = parsed.path.strip('/').split('/')
            if len(path_parts) >= 3 and path_parts[0] == "top-websites":
                folder_category = path_parts[1].replace('-', ' ')
                folder_subcategory = path_parts[2].replace('-', ' ')
            else:
                folder_category = main_cat
                folder_subcategory = subcat

            # Create folder for this category under today's folder
            category_folder = os.path.join(today_folder, folder_category)
            os.makedirs(category_folder, exist_ok=True)
            file_name = f"{folder_subcategory}.csv"

            # Scrape table data and save to CSV
            if scrape_similarweb_data_page(driver, current_url, category_folder, file_name):
                scraped_count += 1
                print(f"Successfully scraped data for {main_cat} > {subcat}")
            else:
                print(f"Failed to scrape data for {main_cat} > {subcat}")
                break # No retry for scraping function failure, move to next category

            # Delay between requests to avoid overloading the server
            time.sleep(5)
            break # If successful, break out of retry loop and move to next category

        except Exception as e:
            if attempt <= retries:
                print(f"Error processing category {main_cat} > {subcat}: {e}. Retrying in 10 seconds (Attempt {attempt}/{retries+1})...")
                time.sleep(10)
                driver.quit() # Quit and re-init driver before retry to handle potential driver issues
                driver = init_driver()
                driver.get(top_websites_url) # Navigate back to the page
                # You might need to handle cookie banner again if it reappears after driver re-init and navigation
                try:
                    cookie_button = driver.find_element(By.CSS_SELECTOR, "button.accept-cookies, #accept-cookies")
                    cookie_button.click()
                    print("Cookie banner dismissed again after driver restart.")
                except NoSuchElementException:
                    print("No cookie banner found after driver restart.")

                continue # Go to the next iteration of the while loop (retry same category)
            else:
                print(f"Error processing category {main_cat} > {subcat}: {e}. Retry failed after {retries+1} attempts. Moving to the next category.")
                driver.quit() # Quit and re-init driver after final failure before next category
                driver = init_driver()
                driver.get(top_websites_url) # Navigate back to the page for next category
                 # You might need to handle cookie banner again
                try:
                    cookie_button = driver.find_element(By.CSS_SELECTOR, "button.accept-cookies, #accept-cookies")
                    cookie_button.click()
                    print("Cookie banner dismissed again after driver restart for next category.")
                except NoSuchElementException:
                    print("No cookie banner found after driver restart for next category.")
                break # Break out of retry loop and move to next category


print(f"Successfully scraped {scraped_count} categories.")

Page body loaded.
No cookie banner found.
Selected category option: Arts & Entertainment > Animation and Comics
Clicked 'Go' button.
Table data loaded.
Data saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\raw data\03-14-2025\arts and entertainment\animation and comics.csv
Successfully scraped data for Arts & Entertainment > Animation and Comics
Selected category option: Arts & Entertainment > Arts and Entertainment - Other
Clicked 'Go' button.
Table data loaded.
Data saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\raw data\03-14-2025\arts and entertainment\arts and entertainment.csv
Successfully scraped data for Arts & Entertainment > Arts and Entertainment - Other
Selected category option: Arts & Entertainment > Books and Literature
Clicked 'Go' button.
Table data loaded.
Data saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\raw data\03-14-2025\arts and entertainment\books and literature.csv
Successfully scraped data for Arts & Entertainment

## Data validation

In [15]:
def validate_data(base_folder, expected_count, expected_columns):
    """
    Validate that the number of CSV files in base_folder equals expected_count,
    and that each CSV file contains all expected columns.
    
    Returns:
      True if validation passes; otherwise, False.
    """
    csv_files = []
    for root, _, files in os.walk(base_folder):
        for file in files:
            if file.endswith('.csv'):
                csv_files.append(os.path.join(root, file))
    if len(csv_files) != expected_count:
        print(f"Validation error: Expected {expected_count} CSV files, found {len(csv_files)}.")
        return False

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            if set(expected_columns) - set(df.columns):
                print(f"Validation error: Missing columns in {csv_file}")
                return False
            if df.isnull().values.any():
                print(f"Validation error: Missing Values in {csv_file}")
                return False
        except Exception as e:
            print(f"Error reading {csv_file}: {e}")
            return False

    print("Data validation successful.")
    return True

In [ ]:
expected_columns = ['Rank', 'Website', 'Category', 'Rank Change',
                    'Avg. Visit Duration', 'Pages / Visit', 'Bounce Rate']
if validate_data(today_folder, scraped_count, expected_columns):
    print("All scraped data validated successfully.")
else:
    print("Data validation failed.")

driver.quit()

Validation error: Expected 187 CSV files, found 186.
Data validation failed.


# **Part 2:** Data Processing

## Combine The Data

In [21]:
def combine_csv_files(base_folder, exclude_folders=None):
    """
    Combine all CSV files in the base_folder into a single DataFrame.
    
    Args:
        base_folder (str): The base directory containing CSV files.
        exclude_folders (list): List of folder names to exclude.
        
    Returns:
        pd.DataFrame: Combined DataFrame.
    """
    if exclude_folders is None:
        exclude_folders = []
    
    combined_df = pd.DataFrame()
    
    for root, dirs, files in os.walk(base_folder):
        # Use the current folder name for identification
        current_folder = os.path.basename(root)
        if current_folder in exclude_folders:
            print(f"Excluding folder: {root}")
            continue
        
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                df = pd.read_csv(file_path)
                # Add a column to identify the source folder (category)
                df['Fetched From'] = current_folder
                combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df

In [22]:
# Define the base data folder and today's folder (formatted as MM-DD-YYYY)
base_data_folder = os.path.join(os.getcwd(), "raw data")
today_str = datetime.datetime.now().strftime("%m-%d-%Y")
today_folder = os.path.join(base_data_folder, today_str)

# Combine all CSV files from today's folder
grouped_data = combine_csv_files(today_folder)
print(f"Combined CSV files shape: {grouped_data.shape}")

Combined CSV files shape: (9300, 8)


## Compute Missing Value Percentage

In [23]:
def missing_value_percentage(df):
    """
    Calculate the percentage of missing values for each column.
    
    Args:
        df (pd.DataFrame): Input DataFrame.
        
    Returns:
        pd.DataFrame: Missing value percentages per column.
    """
    missing_percent = df.isnull().sum() / len(df) * 100
    missing_df = pd.DataFrame({'Missing Values (%)': missing_percent})
    return missing_df

In [24]:

missing_df = missing_value_percentage(grouped_data)
print("Missing Value Percentage:")
print(missing_df)

Missing Value Percentage:
                     Missing Values (%)
Rank                                0.0
Website                             0.0
Category                            0.0
Rank Change                         0.0
Avg. Visit Duration                 0.0
Pages / Visit                       0.0
Bounce Rate                         0.0
Fetched From                        0.0


## Save Combined Data

In [43]:
# Save the combined DataFrame to a CSV file in today's folder
combined_csv_path = os.path.join(os.getcwd(),"processed data", "full_data.csv")

# Create the directory if it does not exist
os.makedirs(os.path.join(os.getcwd(),"processed data"), exist_ok=True)

grouped_data.to_csv(combined_csv_path, index=False)
print(f"Combined data saved to {combined_csv_path}")

Combined data saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\processed data\full_data.csv


Great news! The data is now ready for use. This process took a bit of time, as I encountered some missing columns. After an hour of troubleshooting, I discovered that certain columns can disappear at specific sizes, which can be tricky to manage, especially in headless mode.

If you happen to have your IP address blocked (which isn’t an issue for me), I recommend trying the Tor network or using a proxy to help with that.

## Formatting Date for AVG Session TIme

In [26]:
def convert_to_minutes(time_str):
    if time_str == '--':  # Handle invalid entries
        return None
    try:
        parts = time_str.split(':')
        if len(parts) != 3:
            return None
        hours, minutes, seconds = map(int, parts)
        return hours * 60 + minutes + seconds / 60
    except ValueError:
        return None

# Apply the conversion to the "Avg. Visit Duration" column
grouped_data['AVG minutes'] = grouped_data['Avg. Visit Duration'].apply(convert_to_minutes)
print("First 3 rows after converting Avg. Visit Duration to minutes:")
print(grouped_data.head(3))

First 3 rows after converting Avg. Visit Duration to minutes:
   Rank       Website Category Rank Change Avg. Visit Duration  Pages / Visit  \
0     1   pornhub.com    Adult           =            00:07:31           7.63   
1     2   xvideos.com    Adult           =            00:10:07           9.53   
2     3  xhamster.com    Adult           =            00:08:34           7.35   

  Bounce Rate Fetched From  AVG minutes  
0      26.61%        Adult     7.516667  
1      23.95%        Adult    10.116667  
2      27.89%        Adult     8.566667  


In [27]:
# Check rows with missing "AVG minutes"
missing_avg = grouped_data[grouped_data['AVG minutes'].isna()]
print("Rows with missing 'AVG minutes':")
missing_avg

Rows with missing 'AVG minutes':


,Rank,Website,Category,Rank Change,Avg. Visit Duration,Pages / Visit,Bounce Rate,Fetched From,AVG minutes
775,26,narutouacjp.sharepoint.com,Business and Consumer Services > Moving & Relo...,1,--,1.67,5.58%,business and consumer services,NaN
791,42,billfish-security.squarespace.com,Business and Consumer Services > Moving & Relo...,4,--,1.10,57.92%,business and consumer services,NaN
920,21,superricascol-my.sharepoint.com,Community and Society > Community and Society ...,=,--,4.79,28.66%,community and society,NaN
934,35,are-you-wet-blog.tumblr.com,Community and Society > Community and Society ...,11,--,1.97,65.21%,community and society,NaN
939,40,betweendaylightanddarkness.wordpress.com,Community and Society > Community and Society ...,=,--,1.00,98.93%,community and society,NaN
941,42,krigercro.wordpress.com,Community and Society > Community and Society ...,=,--,1.08,96.97%,community and society,NaN
942,43,thesecrazytimesblog.wordpress.com,Community and Society > Community and Society ...,16,--,1.00,92.63%,community and society,NaN
945,46,basicapparel2022.myshopify.com,Community and Society > Community and Society ...,=,--,1.00,98.2%,community and society,NaN
947,48,handlezen.wordpress.com,Community and Society > Community and Society ...,19,--,1.00,99.1%,community and society,NaN
949,50,lightofangels.weebly.com,Community and Society > Community and Society ...,19,--,1.02,94.97%,community and society,NaN


In [28]:
# Group by 'Category' and calculate mean visit duration (in minutes)
def cal_category_avg_min(df):
    mean_visit_duration = df.groupby('Category')['AVG minutes'].mean().reset_index()
    mean_visit_duration = mean_visit_duration.sort_values(by='AVG minutes')
    print("Mean visit duration by category:")
    print(mean_visit_duration)
    return mean_visit_duration

cal_category_avg_min(grouped_data)

Mean visit duration by category:
                                              Category  AVG minutes
144                 Science and Education > Literature     1.455782
132  Reference Materials > Public Records and Direc...     1.965000
92   Heavy Industry and Engineering > Waste Water a...     2.021333
184                             Vehicles > Motorsports     2.099667
119                      Lifestyle > Lifestyle - Other     2.127333
..                                                 ...          ...
35   Computers Electronics and Technology > Social ...     7.409667
59                         Gambling > Gambling - Other     8.632000
72                Health > Dentist and Dental Services    11.219667
1          Arts & Entertainment > Animation and Comics    15.880667
3          Arts & Entertainment > Books and Literature    16.310000

[186 rows x 2 columns]


,Category,AVG minutes
144,Science and Education > Literature,1.455782
132,Reference Materials > Public Records and Direc...,1.965000
92,Heavy Industry and Engineering > Waste Water a...,2.021333
184,Vehicles > Motorsports,2.099667
119,Lifestyle > Lifestyle - Other,2.127333
...,...,...
35,Computers Electronics and Technology > Social ...,7.409667
59,Gambling > Gambling - Other,8.632000
72,Health > Dentist and Dental Services,11.219667
1,Arts & Entertainment > Animation and Comics,15.880667


## Drop unwanted columns and remove duplicates

In [29]:
cols_to_drop = ['Rank Change', 'Pages / Visit', 'Bounce Rate', 'Fetched From', 'Avg. Visit Duration']
print(f"Number of Rows Before Removing Duplicates: {len(grouped_data)}")
grouped_data.drop(cols_to_drop, axis=1, inplace=True)
grouped_data = grouped_data.drop_duplicates()
print(f"Number of Rows After Removing Duplicates: {len(grouped_data)}")

Number of Rows Before Removing Duplicates: 9300
Number of Rows After Removing Duplicates: 9300


In [33]:
def remove_category_vals(df, values):
    initial_row_count = len(df)
    print(f"Before Removing Categories, Number of Rows: {initial_row_count}")
    for value in values:
        df = df[~df['Category'].str.contains(value, regex=True, na=False)]
        print(f"After Removing '{value}', Number of Rows: {len(df)}")
    final_row_count = len(df)
    print(f"Final Number of Rows After All Removals: {final_row_count}")
    return df

# Focus on Most Important Categories
categories_to_remove = [
    'Adult','Vehicles', 'Car Rentals', 'LGBTQ', 'Dating and Relationships', 'Weather', 'Holidays and Seasonal Events',
    'Universities and Colleges', 'Decease', 'Philanthropy', 'Telecommunications', 'Tickets', 'Libraries and Museums',
    'Gambling', 'Hobbies and Leisure - Other', 'Weddings', 'Advertising Networks', 'Models', 'Law and Government',
    'Coupons and Rebates', 'Tobacco', 'Lifestyle - Other', 'Heavy Industry and Engineering',
    'Camping Scouting and Outdoors', 'Classifieds', 'Community and Society',
    'Maps', 'Reference Materials > Public Records and Directories', 'Pets and Animals > Animals',
    'Pets and Animals > Birds', 'Pets and Animals > Fish and Aquaria', 'Pets and Animals > Horses',
    'Pets and Animals > Pets and Animals - Other', 'Pets and Animals > Pets', 'Pets and Animals'
]
grouped_data = remove_category_vals(grouped_data, categories_to_remove)
cal_category_avg_min(grouped_data).head(5)

Before Removing Categories, Number of Rows: 6350
After Removing 'Adult', Number of Rows: 6300
After Removing 'Vehicles', Number of Rows: 6300
After Removing 'Car Rentals', Number of Rows: 6300
After Removing 'LGBTQ', Number of Rows: 6300
After Removing 'Dating and Relationships', Number of Rows: 6300
After Removing 'Weather', Number of Rows: 6300
After Removing 'Holidays and Seasonal Events', Number of Rows: 6300
After Removing 'Universities and Colleges', Number of Rows: 6300
After Removing 'Decease', Number of Rows: 6300
After Removing 'Philanthropy', Number of Rows: 6300
After Removing 'Telecommunications', Number of Rows: 6300
After Removing 'Tickets', Number of Rows: 6300
After Removing 'Libraries and Museums', Number of Rows: 6300
After Removing 'Gambling', Number of Rows: 6300
After Removing 'Hobbies and Leisure - Other', Number of Rows: 6300
After Removing 'Weddings', Number of Rows: 6300
After Removing 'Advertising Networks', Number of Rows: 6300
After Removing 'Models', Numbe

,Category,AVG minutes
94,Science and Education > Literature,1.455782
103,Sports > Boxing,2.191667
96,Science and Education > Philosophy,2.325667
57,Health > Health Conditions and Concerns,2.339000
70,Home and Garden > Gardening,2.340667


## Check Website URL Availability

In [34]:
def check_url_availability(row):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8'
    }
    url = row['Website']
    try:
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        response = requests.get(url, headers=headers, timeout=10)
        if 200 <= response.status_code < 300 or response.status_code in [403, 406]:
            return row
        else:
            print(f"Removed: {url} (Status code: {response.status_code})")
            return None
    except requests.RequestException as e:
        print(f"Removed: {url} (Error: {e})")
        return None

def check_urls_in_parallel(df):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(executor.map(check_url_availability, df.to_dict('records')))
    return [result for result in results if result is not None]

valid_results = check_urls_in_parallel(grouped_data.reset_index())
valid_df = pd.DataFrame(valid_results)
valid_csv_path = os.path.join(today_folder, "websites_with_availability.csv")
valid_df.to_csv(valid_csv_path, index=False)
print(f"Websites with availability saved to {valid_csv_path}")

Removed: http://mangaraw.best (Error: HTTPConnectionPool(host='mangaraw.best', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x0000024D70AA2F50>: Failed to resolve 'mangaraw.best' ([Errno 11004] getaddrinfo failed)")))
Removed: http://cmoa.jp (Error: HTTPConnectionPool(host='cmoa.jp', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x0000024D70D67D50>: Failed to resolve 'cmoa.jp' ([Errno 11001] getaddrinfo failed)")))
Removed: http://miruro.tv (Error: HTTPConnectionPool(host='miruro.tv', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x0000024D70D67C50>: Failed to resolve 'miruro.tv' ([Errno 11004] getaddrinfo failed)")))
Removed: http://syosetu.is (Error: HTTPConnectionPool(host='syosetu.is', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<ur

## Grouping The Data

In [35]:
valid_df['Category'].unique()

array(['Arts & Entertainment > Animation and Comics',
       'Arts & Entertainment > Arts and Entertainment - Other',
       'Arts & Entertainment > Books and Literature',
       'Arts & Entertainment > Humor', 'Arts & Entertainment > Music',
       'Arts & Entertainment > Performing Arts',
       'Arts & Entertainment > Streaming & Online TV',
       'Arts & Entertainment > Visual Arts and Design',
       'Business and Consumer Services > Business and Consumer Services - Other',
       'Business and Consumer Services > Business Services',
       'Business and Consumer Services > Marketing and Advertising',
       'Business and Consumer Services > Digital Marketing',
       'Business and Consumer Services > Printing & Self Publishing',
       'Business and Consumer Services > Real Estate',
       'Business and Consumer Services > Moving & Relocation',
       'Business and Consumer Services > Shipping and Logistics',
       'Business and Consumer Services > Textiles',
       'Computers 

We will categorize the data into the following groups:
*   Arts & Entertainment
*   Business & Marketing
*   Technology
*   Tools
*   Shopping
*   Finance
*   Food & Health
*   Games
*   Lifestyle & Home
*   Jobs & Career
*   Sports & News
*   Travel

In [37]:
category_mappings = {

    "Arts & Entertainment" : ['Arts & Entertainment > Animation and Comics',
 'Arts & Entertainment > Arts and Entertainment - Other',
 'Arts & Entertainment > Books and Literature',
 'Arts & Entertainment > Humor',
 'Arts & Entertainment > Music',
 'Arts & Entertainment > Performing Arts',
 'Arts & Entertainment > Streaming & Online TV',
 'Arts & Entertainment > Visual Arts and Design',
],
    
    "Business & Consumer Services" : [ 'Business and Consumer Services > Business Services',
 'Business and Consumer Services > Business and Consumer Services - Other',
 'Business and Consumer Services > Digital Marketing',
 'Business and Consumer Services > Marketing and Advertising',
 'Business and Consumer Services > Moving & Relocation',
 'Business and Consumer Services > Printing & Self Publishing',
 'Business and Consumer Services > Real Estate',
 'Business and Consumer Services > Shipping and Logistics',
 'Business and Consumer Services > Textiles','Business and Consumer Services'],
    
    
    "Technology" : ['Computers Electronics and Technology > Computer Hardware',
 'Computers Electronics and Technology > Computer Security',
'Computers Electronics and Technology > Consumer Electronics'],
    
    
    "Tools" : ['Computers Electronics and Technology > Email',
 'Computers Electronics and Technology > File Sharing and Hosting',
'Computers Electronics and Technology > Graphics Multimedia and Web Design',
'Computers Electronics and Technology > Programming and Developer Software',
 'Computers Electronics and Technology > Search Engines',
'Computers Electronics and Technology > Web Hosting and Domain Names',
'Computers Electronics and Technology > Computers Electronics and Technology - Other'],
    
    "Social Media" : ['Computers Electronics and Technology > Social Media Networks',],
    
    "Shopping" : [ 'Ecommerce & Shopping > Auctions',
 'Ecommerce & Shopping > Ecommerce and Shopping - Other',
 'Ecommerce & Shopping > Marketplace',
 'Ecommerce & Shopping > Price Comparison',
 'Pets and Animals > Pet Food and Supplies',],
    
    "Finance" : [ 'Finance > Accounting and Auditing',
 'Finance > Banking Credit and Lending',
 'Finance > Finance - Other',
 'Finance > Financial Planning and Management',
 'Finance > Insurance',
 'Finance > Investing',],
    
    "Health & Food" : [ 'Food and Drink > Beverages',
 'Food and Drink > Cooking and Recipes',
 'Food and Drink > Food and Drink - Other',
 'Food and Drink > Groceries',
 'Food and Drink > Restaurants and Delivery',
 'Food and Drink > Vegetarian and Vegan',
 'Health > Addictions',
 'Health > Alternative and Natural Medicine',
 'Health > Biotechnology and Pharmaceuticals',
 'Health > Childrens Health',
 'Health > Dentist and Dental Services',
 'Health > Developmental and Physical Disabilities',
 'Health > Geriatric and Aging Care',
 'Health > Health - Other',
 'Health > Health Conditions and Concerns',
 'Health > Medicine',
 'Health > Mens Health',
 'Health > Mental Health',
 'Health > Nutrition Diets and Fitness',
 'Health > Pharmacy',
 'Health > Public Health and Safety',
 'Health > Womens Health',],
    
    "Games" : [ 'Games > Board and Card Games',
 'Games > Games - Other',
 'Games > Puzzles and Brainteasers',
 'Games > Roleplaying Games',
 'Games > Video Games Consoles and Accessories',],
    
    
    "Life Style & Hobbies" : [ 'Hobbies and Leisure > Ancestry and Genealogy',
 'Hobbies and Leisure > Antiques and Collectibles',
 'Hobbies and Leisure > Crafts',
 'Hobbies and Leisure > Photography',
 'Hobbies and Leisure',
'Lifestyle > Beauty and Cosmetics',
 'Lifestyle > Childcare',
 'Lifestyle > Fashion and Apparel',
 'Lifestyle > Gifts and Flowers',
 'Lifestyle > Jewelry and Luxury Products',],
    
    'Home & Garden': [
'Home and Garden > Furniture',
 'Home and Garden > Gardening',
 'Home and Garden > Home Improvement and Maintenance',
 'Home and Garden > Home and Garden - Other',
 'Home and Garden > Interior Design', 'Home and Garden',
    ],
    
    "Jobs & Careers" : [ 'Jobs and Career > Human Resources',
 'Jobs and Career > Jobs and Career - Other',
 'Jobs and Career > Jobs and Employment',],
    
    "Science & Education" : [ 'Reference Materials > Dictionaries and Encyclopedias',
 'Reference Materials > Reference Materials - Other',
 'Science and Education > Astronomy',
 'Science and Education > Biology',
 'Science and Education > Business Training',
 'Science and Education > Chemistry',
 'Science and Education > Earth Sciences',
 'Science and Education > Education',
 'Science and Education > Environmental Science',
 'Science and Education > Grants Scholarships and Financial Aid',
 'Science and Education > History',
 'Science and Education > Literature',
 'Science and Education > Math',
 'Science and Education > Philosophy',
 'Science and Education > Physics',
 'Science and Education > Public Records and Directories',
 'Science and Education > Science and Education - Other',
 'Science and Education > Social Sciences','Science and Education'],
    
    "News & Sport" : [ 'Sports > American Football',
 'Sports > Baseball',
 'Sports > Basketball',
 'Sports > Boxing',
 'Sports > Climbing',
 'Sports > Cycling and Biking',
 'Sports > Extreme Sports',
 'Sports > Fantasy Sports',
 'Sports > Fishing',
 'Sports > Golf',
 'Sports > Hunting and Shooting',
 'Sports > Martial Arts',
 'Sports > Rugby',
 'Sports > Running',
 'Sports > Soccer',
 'Sports > Sports - Other',
 'Sports > Tennis',
 'Sports > Volleyball',
 'Sports > Water Sports',
 'Sports > Winter Sports',
  'News & Media Publishers','Sports' ],
    
"Travel": ['Travel and Tourism > Accommodation and Hotels',
   'Travel and Tourism > Travel and Tourism - Other',
   'Travel and Tourism > Ground Transportation', 'Travel and Tourism',
   'Travel and Tourism > Air Travel',
   'Travel and Tourism > Tourist Attractions',
   'Travel and Tourism > Transportation and Excursions']
    
}

In [38]:
def swap_categories(df, values_to_replace, target_value):
    # Replace specified values with the target value
    df['Category'] = df['Category'].replace(values_to_replace, target_value, inplace=False)
    return df

In [45]:
final_df = valid_df.copy()
for target_category, values in category_mappings.items():
    final_df = swap_categories(final_df, values, target_category)

In [46]:
final_df['Category'].unique()

array(['Arts & Entertainment', 'Business & Consumer Services',
       'Technology', 'Tools', 'Social Media', 'Shopping', 'Finance',
       'Health & Food', 'Games', 'Life Style & Hobbies', 'Home & Garden',
       'Jobs & Careers', 'News & Sport', 'Science & Education', 'Travel'],
      dtype=object)

## Save Final Processed Data

In [47]:
# Create the Folder if Not Exists
os.makedirs(os.path.join(os.getcwd(), "processed data"), exist_ok=True)

# Save the final processed DataFrame to a CSV file in today's folder
final_csv_path = os.path.join(os.getcwd(),"processed data", "full_data_processed.csv")
final_df.to_csv(final_csv_path, index=False)
print(f"Final processed data saved to {final_csv_path}")

# Save the final DataFrame as a JSON file with 'Website' as the index
final_df.set_index('Website', inplace=True)
json_path = os.path.join(os.getcwd(),"processed data", "Web Classification.json")
final_df.to_json(json_path, orient='index')
print(f"Final JSON saved to {json_path}")

Final processed data saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\processed data\full_data_processed.csv
Final JSON saved to c:\Users\Ahmed Kamal\Desktop\TimJS\TimJS\WebScrapping\processed data\Web Classification.json
